In [1]:
import pandas as pd

# Load Bronze
df = pd.read_parquet(r"../data/bronze/yellow_tripdata_2026-04.parquet")

# Cleaning
df_silver = df.copy()
df_silver['trip_duration'] = (
    df_silver['tpep_dropoff_datetime'] - df_silver['tpep_pickup_datetime']
).dt.total_seconds() / 60

df_silver = df_silver.dropna(subset=['passenger_count', 'RatecodeID', 'store_and_fwd_flag'])
df_silver = df_silver[df_silver['trip_duration'] > 1]
df_silver = df_silver[df_silver['trip_duration'] < 180]
df_silver = df_silver[df_silver['trip_distance'] > 0]
df_silver = df_silver[df_silver['trip_distance'] < 50]
df_silver = df_silver[df_silver['fare_amount'] > 0]
df_silver = df_silver[df_silver['passenger_count'] > 0]

# Feature Engineering
df_silver['pickup_hour'] = df_silver['tpep_pickup_datetime'].dt.hour
df_silver['pickup_day'] = df_silver['tpep_pickup_datetime'].dt.dayofweek
df_silver['pickup_month'] = df_silver['tpep_pickup_datetime'].dt.month
df_silver['is_rush_hour'] = df_silver['pickup_hour'].apply(
    lambda x: 1 if (7 <= x <= 9) or (17 <= x <= 19) else 0
)

print("Shape setelah cleaning:", df_silver.shape)

# Simpan Silver
df_silver.to_parquet(r"../data/silver/yellow_tripdata_silver.parquet", index=False)
print("✅ Silver layer selesai!")

Shape setelah cleaning: (2911106, 25)
✅ Silver layer selesai!
